<a href="https://colab.research.google.com/github/Karna2006/RAG-Basics/blob/main/Rag_visualize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#INSTALL
!pip install umap-learn sentence-transformers plotly \
            scikit-learn pymupdf -q

import fitz
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.preprocessing import normalize
from google.colab import files

# Load both models — this is the comparison
# MiniLM:  80MB,  384 dimensions, fast
# MPNet:  420MB, 768 dimensions, higher quality
print("Loading models...")
model_mini  = SentenceTransformer("all-MiniLM-L6-v2")
model_mpnet = SentenceTransformer("all-mpnet-base-v2")
print("Both models ready.")

In [11]:
#CELL 2 — create labelled chunks from your PDF (or use built-in sample)
# from google.colab import files

# # ── Option A: use your own PDF ─────────────────────────────────
# uploaded = files.upload()
# pdf_path = list(uploaded.keys())[0]

# def pdf_to_labelled_chunks(path, chunk_size=300, max_chunks=50):
#     doc   = fitz.open(path)
#     pages = [page.get_text() for page in doc]
#     chunks, labels = [], []

#     for page_num, text in enumerate(pages):
#         if len(chunks) >= max_chunks: break
#         start = 0
#         while start < len(text) and len(chunks) < max_chunks:
#             chunk = text[start : start + chunk_size].strip()
#             if len(chunk) > 40:
#                 chunks.append(chunk)
#                 labels.append(f"Page {page_num+1}")
#             start += chunk_size - 40

#     print(f"{len(chunks)} chunks from {len(pages)} pages")
#     return chunks, labels

# chunks, page_labels = pdf_to_labelled_chunks(pdf_path)

# ── Option B: no PDF? use this built-in sample ─────────────────
# Chunks deliberately span 5 topics so clusters are visible
SAMPLE_CHUNKS = [
    # --- topic: machine learning ---
    "Gradient descent optimizes the loss function by computing partial derivatives.",
    "Backpropagation computes gradients layer by layer using the chain rule.",
    "Overfitting occurs when a model memorizes training data but fails to generalize.",
    "Regularization techniques like dropout prevent overfitting in deep networks.",
    "Cross-validation splits data to give an unbiased estimate of model performance.",
    "Batch normalization stabilizes training by normalizing layer inputs.",
    "Learning rate schedules reduce the step size as training progresses.",
    "Convolutional layers detect spatial hierarchies in image data.",
    "Attention mechanisms allow models to focus on relevant input positions.",
    "Transfer learning reuses pretrained weights for new downstream tasks.",
    # --- topic: databases ---
    "SQL joins combine rows from two or more tables based on a related column.",
    "Indexes speed up data retrieval but slow down write operations.",
    "ACID properties ensure reliable database transactions.",
    "NoSQL databases sacrifice strict consistency for horizontal scalability.",
    "Vector databases store embeddings and support approximate nearest-neighbor search.",
    "Query planners optimize SQL execution using statistics about table cardinality.",
    "Sharding partitions a database horizontally across multiple nodes.",
    "Foreign keys enforce referential integrity between related tables.",
    "Window functions compute values across a set of rows related to the current row.",
    "Connection pooling reuses database connections to reduce overhead.",
    # --- topic: climate ---
    "Rising CO2 concentrations trap heat and raise global average temperatures.",
    "Arctic ice melt accelerates warming through the ice-albedo feedback loop.",
    "Ocean acidification threatens coral reefs as seas absorb excess carbon dioxide.",
    "Renewable energy sources like solar and wind produce no direct emissions.",
    "Carbon capture technology extracts CO2 from the atmosphere or emission sources.",
    "Permafrost thaw releases methane, a potent greenhouse gas.",
    "Sea level rise threatens low-lying coastal communities worldwide.",
    "Deforestation reduces the planet's ability to absorb carbon emissions.",
    "Extreme weather events become more frequent as the climate warms.",
    "International agreements aim to limit warming to 1.5 degrees Celsius.",
    # --- topic: cooking ---
    "Maillard reaction browns food and creates complex flavours at high heat.",
    "Emulsification combines fat and water using an emulsifier like lecithin.",
    "Fermentation uses microorganisms to transform sugars into alcohol or acid.",
    "Sous vide cooking seals food in bags and cooks at precise low temperatures.",
    "Caramelization occurs when sugar molecules break down above 160 degrees.",
    "Gluten develops when flour proteins hydrate and align during kneading.",
    "Basting keeps meat moist by redistributing surface juices during roasting.",
    "Blanching briefly boils vegetables then shocks them in ice water to halt cooking.",
    "Reduction concentrates flavor by simmering liquid until much of it evaporates.",
    "Brining improves moisture retention in lean meats through osmosis.",
    # --- topic: finance ---
    "Compound interest grows exponentially because returns are reinvested over time.",
    "Diversification reduces portfolio risk by spreading investments across assets.",
    "Options give the holder the right but not the obligation to buy or sell.",
    "Inflation erodes purchasing power when the price level rises over time.",
    "Yield curves show the relationship between bond maturity and interest rates.",
    "Market capitalization equals share price multiplied by total shares outstanding.",
    "Short selling profits from a decline by borrowing and selling shares.",
    "Dollar-cost averaging invests fixed amounts at regular intervals.",
    "Beta measures a stock's volatility relative to the broader market.",
    "ETFs track an index and trade on exchanges like individual stocks.",
]
SAMPLE_LABELS = ["ML"]*10 + ["DB"]*10 + ["Climate"]*10 + \
                ["Cooking"]*10 + ["Finance"]*10

# Use sample if no PDF uploaded
if 'chunks' not in dir():
    chunks, page_labels = SAMPLE_CHUNKS, SAMPLE_LABELS
    print("Using built-in 50-chunk sample (5 topics × 10 chunks)")

topic_labels = SAMPLE_LABELS  # for colour coding

In [ ]:
#CELL 3 — embed with both models, reduce to 2D with UMAP
print("Embedding with MiniLM...")
vecs_mini  = model_mini.encode(chunks, show_progress_bar=True)

print("Embedding with MPNet...")
vecs_mpnet = model_mpnet.encode(chunks, show_progress_bar=True)

print(f"MiniLM shape: {vecs_mini.shape}")   # (50, 384)
print(f"MPNet  shape: {vecs_mpnet.shape}")  # (50, 768)

# ── UMAP: compress N-dimensional vectors → 2D for plotting ────
# n_neighbors: how many nearby points each point considers
#   low  (5)  → very local structure, tight islands
#   high (50) → more global structure, spread out
# min_dist: how tightly points are packed in 2D
#   low  (0.0) → dense clumps
#   high (0.5) → more uniform spread
# random_state: fixes the layout so it's reproducible

reducer = UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.1,
    metric="cosine",    # matches how RAG measures similarity
    random_state=42
)

print("Running UMAP on MiniLM vectors...")
umap_mini  = reducer.fit_transform(normalize(vecs_mini))

print("Running UMAP on MPNet vectors...")
umap_mpnet = reducer.fit_transform(normalize(vecs_mpnet))

# ── Build dataframes for plotting ─────────────────────────────
def make_df(umap_2d, model_name):
    return pd.DataFrame({
        "x":       umap_2d[:, 0],
        "y":       umap_2d[:, 1],
        "topic":   topic_labels,
        "text":    [c[:80] + "..." for c in chunks],
        "model":   model_name,
        "chunk_id": range(len(chunks)),
    })

df_mini  = make_df(umap_mini,  "MiniLM-L6 (384d)")
df_mpnet = make_df(umap_mpnet, "MPNet-base (768d)")
print("Ready to plot.")

In [12]:
#CELL 4 — side-by-side UMAP scatter plot (the main visual)
from plotly.subplots import make_subplots
import plotly.graph_objects as go

COLORS = {
    "ML":      "#7F77DD",
    "DB":      "#1D9E75",
    "Climate": "#D85A30",
    "Cooking": "#BA7517",
    "Finance": "#D4537E",
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "all-MiniLM-L6-v2 (384 dims)",
        "all-mpnet-base-v2 (768 dims)"
    ]
)

for col, df in enumerate([df_mini, df_mpnet], 1):
    for topic, color in COLORS.items():
        mask = df["topic"] == topic
        sub  = df[mask]
        fig.add_trace(
            go.Scatter(
                x=sub["x"], y=sub["y"],
                mode="markers+text",
                name=topic,
                marker=dict(size=10, color=color, opacity=0.85,
                             line=dict(width=1, color="white")),
                text=[str(i) for i in sub["chunk_id"]],
                textposition="top center",
                textfont=dict(size=8),
                hovertemplate="<b>%{customdata}</b><extra></extra>",
                customdata=sub["text"],
                legendgroup=topic,
                showlegend=(col == 1),   # legend only once
            ),
            row=1, col=col
        )

fig.update_layout(
    height=520,
    title_text="Embedding space — hover any point to read the chunk",
    legend_title="Topic",
    font=dict(size=12),
    plot_bgcolor="#f8f8f8",
    paper_bgcolor="white",
)
fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False)

fig.show()

# Save as HTML so you can open it outside Colab
fig.write_html("embedding_clusters.html")
print("Saved to embedding_clusters.html")

Saved to embedding_clusters.html


In [13]:
#CELL 5 — cosine similarity heatmap (shows model quality numerically)
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity

# Compute full 50×50 similarity matrices
sim_mini  = cosine_similarity(normalize(vecs_mini))
sim_mpnet = cosine_similarity(normalize(vecs_mpnet))

# Sort chunks by topic so same-topic blocks appear on the diagonal
order = sorted(range(len(topic_labels)), key=lambda i: topic_labels[i])
labels_sorted = [f"{topic_labels[i][:2]}{i}" for i in order]

def reorder(mat, idx):
    return mat[np.ix_(idx, idx)]

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "MiniLM — similarity matrix",
        "MPNet — similarity matrix"
    ]
)

for col, (mat, title) in enumerate([
    (reorder(sim_mini,  order), "MiniLM"),
    (reorder(sim_mpnet, order), "MPNet"),
], 1):
    fig2.add_trace(
        go.Heatmap(
            z=mat,
            x=labels_sorted, y=labels_sorted,
            colorscale="RdBu_r",
            zmin=0, zmax=1,
            showscale=(col == 2),
            hovertemplate="sim: %{z:.3f}<extra></extra>",
        ),
        row=1, col=col
    )

fig2.update_layout(
    height=500,
    title_text="Cosine similarity — bright blocks on diagonal = tight clusters",
    font=dict(size=10),
)
fig2.update_xaxes(showticklabels=False)
fig2.update_yaxes(showticklabels=False)
fig2.show()

# ── Print the numbers that matter ────────────────────────────
topics = list(set(topic_labels))
print(f"\n{'Topic':<10} {'MiniLM intra':>14} {'MPNet intra':>12} {'Better':>8}")
print("-" * 48)

for topic in sorted(topics):
    idx = [i for i, t in enumerate(topic_labels) if t == topic]
    # intra-cluster similarity: avg similarity of same-topic pairs
    mini_intra  = np.mean([sim_mini[i,j]  for i in idx
                             for j in idx if i != j])
    mpnet_intra = np.mean([sim_mpnet[i,j] for i in idx
                             for j in idx if i != j])
    winner = "MPNet" if mpnet_intra > mini_intra else "MiniLM"
    print(f"{topic:<10} {mini_intra:>14.3f} {mpnet_intra:>12.3f} {winner:>8}")

# Also compute inter-cluster separation (lower = better distinction)
all_idx = list(range(len(topic_labels)))
print("\nInter-cluster similarity (lower = cleaner separation):")
for t1 in sorted(topics):
    for t2 in sorted(topics):
        if t1 >= t2: continue
        i1 = [i for i, t in enumerate(topic_labels) if t == t1]
        i2 = [i for i, t in enumerate(topic_labels) if t == t2]
        m = np.mean([sim_mini[i,j]  for i in i1 for j in i2])
        p = np.mean([sim_mpnet[i,j] for i in i1 for j in i2])
        print(f"  {t1} vs {t2}: MiniLM={m:.3f}  MPNet={p:.3f}")


Topic        MiniLM intra  MPNet intra   Better
------------------------------------------------
Climate             0.263        0.346    MPNet
Cooking             0.268        0.316    MPNet
DB                  0.216        0.260    MPNet
Finance             0.207        0.232    MPNet
ML                  0.277        0.326    MPNet

Inter-cluster similarity (lower = cleaner separation):
  Climate vs Cooking: MiniLM=0.067  MPNet=0.095
  Climate vs DB: MiniLM=0.016  MPNet=0.034
  Climate vs Finance: MiniLM=0.044  MPNet=0.081
  Climate vs ML: MiniLM=0.025  MPNet=0.034
  Cooking vs DB: MiniLM=0.034  MPNet=0.056
  Cooking vs Finance: MiniLM=0.035  MPNet=0.057
  Cooking vs ML: MiniLM=0.054  MPNet=0.067
  DB vs Finance: MiniLM=0.077  MPNet=0.079
  DB vs ML: MiniLM=0.084  MPNet=0.103
  Finance vs ML: MiniLM=0.066  MPNet=0.073
